In [0]:
%pip install -q xgboost lightgbm

In [0]:
# =========================
# CONFIGURATION
# =========================
dbutils.widgets.text("TABLE_NAME", "ml.data.model_24h")
dbutils.widgets.text("TARGET_COL", "label_fail_24h")
dbutils.widgets.text("HORIZON_HOURS", "24")
dbutils.widgets.text("MODEL_NAME_PREFIX", "wind_turbine_failure")
dbutils.widgets.text("CATALOG_NAME", "ml")
dbutils.widgets.text("SCHEMA_NAME", "models")
dbutils.widgets.text("EXPERIMENT_NAME", "/Shared/wind_turbine_failure")

# =========================
# DEPENDENCIES
# =========================


import pyspark.sql.functions as F
import mlflow
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_curve
)
import xgboost as xgb
from lightgbm import LGBMClassifier
from mlflow.models.signature import infer_signature

# =========================
# MLFLOW SETUP (CRITICAL)
# =========================
experiment_name = dbutils.widgets.get("EXPERIMENT_NAME")
mlflow.set_experiment(experiment_name)
mlflow.set_registry_uri("databricks-uc")

# =========================
# UTILITIES
# =========================
def load_data(table):
    return spark.table(table)

def get_feature_columns(df, target):
    return [c for c in df.columns if c not in [target, "hour_start", "turbine_id"]]

def time_split(df):
    train = df.filter("hour_start <= '2023-06-30'")
    test  = df.filter("hour_start >= '2023-07-01'")
    return train, test

def compute_scale_pos_weight(df, target):
    neg = df.filter(F.col(target) == 0).count()
    pos = df.filter(F.col(target) == 1).count()
    return neg / max(pos, 1)

def find_best_threshold(y_true, y_prob):
    p, r, t = precision_recall_curve(y_true, y_prob)
    f1 = 2 * (p * r) / (p + r + 1e-9)
    idx = np.argmax(f1)
    return t[idx], f1[idx]

def get_models(scale_pos_weight):
    return {
        "xgboost": xgb.XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            eval_metric="aucpr",
            random_state=42
        ),
        "lightgbm": LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            scale_pos_weight=scale_pos_weight,
            random_state=42
        )
    }

# =========================
# TRAIN + EVALUATE + TRACK
# =========================


def train_and_select_best(train_df, test_df, features, target, models):
    X_train = train_df.select(features).toPandas()
    y_train = train_df.select(target).toPandas().values.ravel()
    X_test  = test_df.select(features).toPandas()
    y_test  = test_df.select(target).toPandas().values.ravel()

    best = {"f1": -1}

    for name, model in models.items():
        with mlflow.start_run(run_name=name) as run:
            model.fit(X_train, y_train)
            probs = model.predict_proba(X_test)[:, 1]

            thr, f1 = find_best_threshold(y_test, probs)
            preds = (probs >= thr).astype(int)

            mlflow.log_metrics({
                "pr_auc": average_precision_score(y_test, probs),
                "precision": precision_score(y_test, preds),
                "recall": recall_score(y_test, preds),
                "f1": f1,
                "threshold": thr
            })

            # Infer signature and provide input example
            signature = infer_signature(X_train, model.predict(X_train))
            input_example = X_train.iloc[[0]]

            mlflow.sklearn.log_model(
                model,
                "model",
                signature=signature,
                input_example=input_example
            )

            if f1 > best["f1"]:
                best = {
                    "f1": f1,
                    "run_id": run.info.run_id
                }

    return best

# =========================
# REGISTER BEST MODEL
# =========================
def register_best_model(best, horizon, prefix, catalog, schema):
    model_name = f"{catalog}.{schema}.{prefix}_{horizon}h"
    model_uri = f"runs:/{best['run_id']}/model"

    mlflow.register_model(
        model_uri=model_uri,
        name=model_name
    )

    print("✅ MODEL REGISTERED SUCCESSFULLY")
    print("Model:", model_name)
    print("F1:", best["f1"])

# =========================
# PIPELINE EXECUTION
# =========================
table   = dbutils.widgets.get("TABLE_NAME")
target  = dbutils.widgets.get("TARGET_COL")
horizon = dbutils.widgets.get("HORIZON_HOURS")
prefix  = dbutils.widgets.get("MODEL_NAME_PREFIX")
catalog = dbutils.widgets.get("CATALOG_NAME")
schema  = dbutils.widgets.get("SCHEMA_NAME")

df = load_data(table)
features = get_feature_columns(df, target)

train_df, test_df = time_split(df)
scale = compute_scale_pos_weight(train_df, target)

models = get_models(scale)
best = train_and_select_best(train_df, test_df, features, target, models)

register_best_model(best, horizon, prefix, catalog, schema)
